# AutoLens Egypt — M1 (Ready to run on Google Colab)

Just select **Runtime → Run all**. The only required input is in cell 2, which asks you to upload the project ZIP file (any name, such as `AutoLens_M1_project_FULL.zip` or `..._UPDATED.zip`).

The notebook automatically detects the folder name and paths — no edits are required.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select the project ZIP file (any name)
print('Uploaded:', list(uploaded.keys()))

In [ ]:
import os, glob, zipfile, shutil
from pathlib import Path

zips = glob.glob('/content/*.zip')
assert zips, 'ZIP file not found — upload it in the cell above'
zpath = max(zips, key=lambda p: os.path.getsize(p))  # the newest/largest ZIP is the project archive

DEST = Path('/content/autolens')
if DEST.exists():
    shutil.rmtree(DEST)
DEST.mkdir()
with zipfile.ZipFile(zpath) as zf:
    zf.extractall(DEST)

# Automatically detect the project folder (inside the ZIP it may be 'Default Project' or 'AutoLens_M1_project')
candidates = [p for p in DEST.iterdir() if p.is_dir() and (p / 'pyproject.toml').exists()]
if not candidates:
    candidates = [p for p in DEST.rglob('pyproject.toml')]
    candidates = [p.parent for p in candidates]
assert candidates, f'pyproject.toml was not found inside the ZIP: {list(DEST.iterdir())[:10]}'
ROOT = candidates[0]
print('Project directory:', ROOT)
print('Contents:', sorted(p.name for p in ROOT.iterdir())[:12])

In [ ]:
import os, subprocess, sys
os.environ['PROJECT_ROOT'] = str(ROOT)
os.environ['RUN_DATE'] = '2026-01-15'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

# Install core packages (images/tests do not require internet)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT)],
                   capture_output=True, text=True)
print('pip rc =', r.returncode)
if r.returncode: print(r.stdout[-1500:], r.stderr[-1500:])
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest', 'pyarrow', 'Pillow'],
                   capture_output=True, text=True)
print('extras rc =', r.returncode)

## Run the pipeline (about 20 seconds without images, or two minutes with images)

The 250 real photos from Wikimedia Commons are already included in `quarantine/` inside the FULL ZIP, so no download is needed.

In [ ]:
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'autolens.pipelines.entrypoints', '--run-date', '2026-01-15'],
    cwd=ROOT, env=os.environ, capture_output=True, text=True,
)
print(r.stdout[-2500:])
if r.returncode: print('STDERR:', r.stderr[-2000:])
assert r.returncode == 0

## Tests (46 tests)

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'],
                   cwd=ROOT, env=os.environ, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode: print(r.stderr[-1500:])
assert r.returncode == 0

## Download the results

In [ ]:
from google.colab import files
for name in ('linkage_metrics.json', 'open_images_attribution.csv', 'alerts.json'):
    p = ROOT / 'outputs' / name
    if p.exists():
        files.download(str(p))
p = ROOT / 'data_labels' / 'human_review_template.csv'
if p.exists():
    files.download(str(p))
print('Download complete')